# import 

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from dotenv import load_dotenv

import os

In [2]:
PROJECT_PATH = Path.cwd().parent

# CSV location
CSV_PATH = PROJECT_PATH / "data" / "raw" / "monthly_performance-2.csv"

# Load data
monthly_performance = pd.read_csv(CSV_PATH)

# Display first records
display(monthly_performance.head(10))

,customer_id,observation_month,months_on_book,dpd_current,payment_made,payment_amount,scheduled_emi,credit_score,credit_score_delta_3m,revolving_utilization,revolving_util_delta_3m,has_defaulted,will_default_original
0,CUST100000,2022-05,1,0,1,1668.61,1668.61,722,2,0.443,0.010,0,0
1,CUST100000,2022-06,2,0,1,1668.61,1668.61,720,0,0.435,0.002,0,0
2,CUST100000,2022-07,3,0,1,1668.61,1668.61,718,-2,0.417,-0.016,0,0
3,CUST100000,2022-08,4,0,1,1668.61,1668.61,720,6,0.421,0.013,0,0
4,CUST100000,2022-09,5,0,1,1668.61,1668.61,721,3,0.427,0.020,0,0
5,CUST100000,2022-10,6,0,1,1668.61,1668.61,718,-9,0.421,-0.019,0,0
6,CUST100000,2022-11,7,0,1,1668.61,1668.61,716,-6,0.422,0.002,0,0
7,CUST100000,2022-12,8,0,1,1668.61,1668.61,712,-12,0.416,-0.016,0,0
8,CUST100000,2023-01,9,0,1,1668.61,1668.61,708,-12,0.398,-0.056,0,0
9,CUST100000,2023-02,10,0,1,1668.61,1668.61,710,6,0.384,-0.040,0,0


In [3]:
PROJECT_PATH = Path.cwd().parent

# CSV location
CSV_PATH = PROJECT_PATH / "data" / "raw" / "origination_data-2.csv"

# Load data
origination_data = pd.read_csv(CSV_PATH)

# Display first records
display(origination_data.head(10))

,customer_id,application_date,age,gender,employment_status,annual_income,monthly_income,loan_amount,interest_rate,loan_term,emi,dti,credit_score_origination,revolving_utilization_origination,credit_inquiries_12m,deterioration_pattern,default_12m,vintage_year,vintage_month
0,CUST100000,2022-04-08,42,Male,Self-Employed,83834.0,6986.17,50000.0,12.33,36,1668.61,0.35,720,0.433,0,stable,0,2022,2022-04
1,CUST100001,2024-04-28,35,Female,Employed,56472.0,4706.00,50000.0,10.98,48,1291.79,0.41,658,0.250,1,gradual_decline,0,2024,2024-04
2,CUST100002,2023-12-19,44,Male,Employed,165696.0,13808.00,50000.0,9.71,84,822.59,0.44,801,0.339,2,stable,0,2023,2023-12
3,CUST100003,2023-04-27,41,Female,Employed,79943.0,6661.92,50000.0,14.42,36,1719.10,0.38,743,0.617,0,gradual_decline,0,2023,2023-04
4,CUST100004,2023-04-20,28,Female,Employed,160442.0,13370.17,52071.0,10.81,36,1700.06,0.10,662,0.215,1,stable,0,2023,2023-04
5,CUST100005,2024-07-30,32,Male,Employed,67403.0,5616.92,50000.0,11.32,36,1644.52,0.28,602,0.514,3,sudden_shock,1,2024,2024-07
6,CUST100006,2022-04-05,28,Female,Employed,84087.0,7007.25,50000.0,11.05,72,952.98,0.30,658,0.499,1,gradual_decline,0,2022,2022-04
7,CUST100007,2024-02-04,26,Female,Employed,93039.0,7753.25,50000.0,10.81,48,1287.67,0.30,686,0.232,0,stable,1,2024,2024-02
8,CUST100008,2022-08-09,56,Female,Employed,72986.0,6082.17,50000.0,11.42,48,1302.50,0.14,700,0.107,2,sudden_shock,0,2022,2022-08
9,CUST100009,2022-04-14,48,Male,Employed,220137.0,18344.75,76387.0,14.92,48,2122.81,0.31,603,0.293,2,stable,0,2022,2022-04


In [5]:
load_dotenv(PROJECT_PATH / ".env")

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME")
)

engine = create_engine(connection_url)

# Test connection
with engine.connect() as connection:
    result = connection.execute(text("SELECT version()"))
    print(result.scalar())

PostgreSQL 18.7 on x86_64-windows, compiled by msvc-19.44.35229, 64-bit


In [6]:
with engine.begin() as connection:
    connection.execute(text("CREATE SCHEMA IF NOT EXISTS raw"))

origination_data.to_sql(
    name="orig_data",
    con=engine,
    schema="raw",
    if_exists="replace",
    index=False,
    chunksize=1000,
    method="multi"
)

print("Data successfully loaded!")

Data successfully loaded!


In [7]:
with engine.begin() as connection:
    connection.execute(text("CREATE SCHEMA IF NOT EXISTS raw"))

monthly_performance.to_sql(
    name="monthl_perf",
    con=engine,
    schema="raw",
    if_exists="replace",
    index=False,
    chunksize=1000,
    method="multi"
)

print("Data successfully loaded!")

Data successfully loaded!
